# NHLA 1-Side Classification

This notebook trains the single-face NHLA grading model. The workflow is intentionally:
1. prepare the extracted face CSV,
2. run Optuna tuning on a short validation pass,
3. train the final 1-side model using the best hyperparameters,
4. save metrics and checkpoints for comparison against the 2-side model.

In [1]:
from pathlib import Path
import json
import sys
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_NAME = "26-8-18 Test"
CSV_PATH = PROJECT_ROOT / "data" / DATASET_NAME / "nhla_dataset" / "nhla_face_labels.csv"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_ROOT / "models" / f"NHLA_Classification_1Side_{RUN_TIMESTAMP}"
OPTUNA_DIR = PROJECT_ROOT / "logs" / "optuna"
TENSORBOARD_DIR = PROJECT_ROOT / "logs" / "tensorboard" / f"NHLA_Classification_1Side_{RUN_TIMESTAMP}"
METRICS_DIR = PROJECT_ROOT / "data" / "metrics" / f"NHLA_Classification_1Side_{RUN_TIMESTAMP}"

# ------------------------------------------------------------------
# Configuration - change these knobs as needed
# ------------------------------------------------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
NUM_EPOCHS = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.2
SEED = 42
CHECKPOINT_INTERVAL = 5

# Optuna tuning controls
OPTUNA_ENABLED = True
OPTUNA_TRIALS = 5
TUNE_EPOCHS = 4
STUDY_NAME = f"nhla_1side_optuna_{RUN_TIMESTAMP}"
OPTUNA_DB_PATH = OPTUNA_DIR / "NHLA_Classification.db"
OPTUNA_SEARCH_SPACE = {
    "lr": {"low": 1e-5, "high": 1e-2, "log": True},
    "batch_size": [4, 8, 16, 32],
    "weight_decay": {"low": 1e-6, "high": 1e-2, "log": True},
    "encoder_name": ["resnet18", "resnet34", "resnet50"],
}

# Run naming
CHECKPOINT_NAME = f"NHLA_1Side_{RUN_TIMESTAMP}"
LOG_NAME = f"NHLA_1Side_{RUN_TIMESTAMP}"
METRICS_RUN_NAME = f"nhla_1side_{RUN_TIMESTAMP}"

# Model config
MODEL_NAME = "NHLA_1Side_ResNet18"
ENCODER_NAME = "resnet18"  # options: resnet18, resnet34, resnet50
PRETRAINED = True

print(f"CSV path: {CSV_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"TensorBoard dir: {TENSORBOARD_DIR}")
print(f"Metrics dir: {METRICS_DIR}")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL}")
print(f"Checkpoint name: {CHECKPOINT_NAME}")
print(f"Log name: {LOG_NAME}")
print(f"Metrics run name: {METRICS_RUN_NAME}")
print("Optuna search space:")

print(json.dumps(OPTUNA_SEARCH_SPACE, indent=2, default=str))
print("Optuna search space:")


print(json.dumps(OPTUNA_SEARCH_SPACE, indent=2, default=str))


CSV path: /workspaces/Oak-Defect-Detection/data/26-8-18 Test/nhla_dataset/nhla_face_labels.csv
Output dir: /workspaces/Oak-Defect-Detection/models/NHLA_Classification_1Side_20260818_195250
TensorBoard dir: /workspaces/Oak-Defect-Detection/logs/tensorboard/NHLA_Classification_1Side_20260818_195250
Metrics dir: /workspaces/Oak-Defect-Detection/data/metrics/NHLA_Classification_1Side_20260818_195250
Checkpoint interval: 5
Checkpoint name: NHLA_1Side_20260818_195250
Log name: NHLA_1Side_20260818_195250
Metrics run name: nhla_1side_20260818_195250
Optuna search space:
{
  "lr": {
    "low": 1e-05,
    "high": 0.01,
    "log": true
  },
  "batch_size": [
    4,
    8,
    16,
    32
  ],
  "weight_decay": {
    "low": 1e-06,
    "high": 0.01,
    "log": true
  },
  "encoder_name": [
    "resnet18",
    "resnet34",
    "resnet50"
  ]
}
Optuna search space:
{
  "lr": {
    "low": 1e-05,
    "high": 0.01,
    "log": true
  },
  "batch_size": [
    4,
    8,
    16,
    32
  ],
  "weight_decay": 

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebooks.NHLA_Classification_1Side import train_1side

result = train_1side(
    csv_path=str(CSV_PATH),
    output_dir=str(OUTPUT_DIR),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    val_split=VAL_SPLIT,
    seed=SEED,
    optuna_trials=OPTUNA_TRIALS if OPTUNA_ENABLED else 0,
    tune_epochs=TUNE_EPOCHS,
    encoder_name=ENCODER_NAME,
    pretrained=PRETRAINED,
    search_space=OPTUNA_SEARCH_SPACE,
    study_name=STUDY_NAME,
    db_path=str(OPTUNA_DB_PATH),
    checkpoint_name=CHECKPOINT_NAME,
    log_name=LOG_NAME,
    metrics_run_name=METRICS_RUN_NAME,
    tensorboard_dir=str(TENSORBOARD_DIR),
    metrics_dir=str(METRICS_DIR),
    checkpoint_interval=CHECKPOINT_INTERVAL,
)

print(json.dumps({
    "best_params": result["best_params"],
    "metrics": result["result"]["metrics"],

    "metrics_path": result["result"].get("metrics_path"),
}, indent=2))

2026-08-18 19:52:55.434294: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-18 19:52:55.496633: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-18 19:52:55.519150: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[I 2026-08-18 19:53:00,590] A new study created in RDB with name: nhla_1side_optuna_20260818_195250
[I 2026-08-18 19:55:50,503] Trial 0 finished with value: 0.3157356572789378 and parameters: {'lr': 0.0001329291894316216, 'batch_size': 4, 'weight_decay': 4.207053950287936e-06, 'encoder_name': 'resnet34'}. Best is trial 0 with value: 0.3157356572789378.
[I 2026-08

Optuna tuning complete for face mode. Best params: {'lr': 8.200518402245828e-05, 'batch_size': 8, 'weight_decay': 9.565499215943819e-05, 'encoder_name': 'resnet34'}
Best 1-side Optuna params: {'lr': 8.200518402245828e-05, 'batch_size': 8, 'weight_decay': 9.565499215943819e-05, 'encoder_name': 'resnet34'}
{
  "best_params": {
    "lr": 8.200518402245828e-05,
    "batch_size": 8,
    "weight_decay": 9.565499215943819e-05,
    "encoder_name": "resnet34"
  },
  "metrics": {
    "accuracy": 0.3698752228163993,
    "balanced_accuracy": 0.30443556592992255,
    "macro_precision": 0.33718825773203337,
    "macro_recall": 0.30443556592992255,
    "macro_f1": 0.308752967043047,
    "weighted_precision": 0.36078094687694756,
    "weighted_recall": 0.3698752228163993,
    "weighted_f1": 0.36079332800540653,
    "micro_f1": 0.3698752228163993,
    "per_class_precision": [
      0.3366013071895425,
      0.42,
      0.17142857142857143,
      0.3081395348837209,
      0.4119601328903654,
      0.375

## Notes

- Optuna is run before the final training step.
- The tuned parameters are saved to `best_params.json` in the output directory.
- Final training uses the best parameters rather than the initial defaults.